# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faheem-danish/internship-starter-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [33]:
from google.colab import userdata
from datasets import load_dataset

token = userdata.get("HF_TOKEN")

print("Token loaded successfully")

Token loaded successfully


In [34]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "FlyRank/internship-warehouse",
    repo_type="dataset",
    token=token
)

for f in files[:20]:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [35]:
from huggingface_hub import hf_hub_download

march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=token
)

print(march_file)

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [36]:
march_df = con.execute(f"""
SELECT *
FROM read_parquet('{march_file}')
USING SAMPLE 10000 ROWS
""").df()

march_df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-13,client_4a18d1793d92fb84,content_c160a4a06deac9da,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-03
1,2026-03-18,client_65de48885f4ef01b,content_31154fbb5c2256fd,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-03
2,2026-03-19,client_625b6439094e23e4,content_f6e2df1590dac200,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-03
3,2026-03-22,client_08a6a72ff48e62c0,content_e98edc929b2e56c5,True,False,False,<NA>,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-10,client_23a62021009f63c4,content_e9b77d1ce3686831,True,True,True,False,15,0,834,...,0,0,0,0,0,0,0,0,0,2026-03


In [37]:
march_df.shape

(10000, 31)

In [38]:
march_df["report_date"].min(), march_df["report_date"].max()

(Timestamp('2026-03-01 00:00:00'), Timestamp('2026-03-31 00:00:00'))

In [39]:
performance

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
## Unit of Analysis + Time Window

One row represents one content item for one client on one day.

The table used is `fact_content_daily_performance`, which contains daily Search Console and Analytics performance measurements.

For this analysis, I use the middle-panel month **March 2026** (`2026-03`) to avoid using the final month as a future outcome window.

The goal is to support ranking webpages that may deserve content refresh.


In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show first few rows

df_sample = performance["train"].select(range(5))

df_sample

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 5
})

In [41]:
print(performance["train"].features)

{'report_date': Value('date32'), 'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'client_has_gsc': Value('bool'), 'client_has_ga4': Value('bool'), 'gsc_data_available': Value('bool'), 'ga4_data_available': Value('bool'), 'gsc_impressions': Value('int64'), 'gsc_clicks': Value('int64'), 'gsc_sum_position': Value('int64'), 'gsc_avg_position': Value('float64'), 'ga4_pageviews': Value('int64'), 'ga4_sessions': Value('int64'), 'ga4_users': Value('int64'), 'ga4_engaged_sessions': Value('int64'), 'ga4_total_engagement_sec': Value('int64'), 'sessions_organic': Value('int64'), 'sessions_direct': Value('int64'), 'sessions_referral': Value('int64'), 'sessions_social': Value('int64'), 'sessions_paid': Value('int64'), 'sessions_ai': Value('int64'), 'ai_chatgpt': Value('int64'), 'ai_perplexity': Value('int64'), 'ai_gemini': Value('int64'), 'ai_copilot': Value('int64'), 'ai_claude': Value('int64'), 'ai_meta': Value('int64'), 'ai_other': Value('int64'), 'scroll_events': Value('in

In [42]:
import duckdb
import pandas as pd

print("DuckDB ready")

DuckDB ready


In [43]:
!pip install duckdb -q

In [9]:
con = duckdb.connect()

con.register("daily_performance", df)

print("Table created")

Table created


In [10]:
grain_check = con.execute("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM daily_performance
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

grain_check

,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
## Fields: Feature / Label / Context / Excluded

### Features
These fields are available before making a refresh decision and can help rank content opportunities:

- `gsc_impressions` — shows search visibility.
- `gsc_clicks` — shows search traffic received.
- `gsc_avg_position` — shows search ranking performance.
- `ga4_sessions` — shows website engagement from sessions.
- `scroll_events` — shows user engagement behavior.

### Label / Proxy
A direct refresh outcome label is not available in this dataset. For this phase, the output is a decision-support priority score based on observed performance signals.

### Context
These fields are used for grouping and analysis, not as model features:

- `client_hash_id`
- `content_hash_id`
- `report_date`

### Excluded
These fields are excluded because they may introduce leakage or are not available at the decision moment:

- Future performance measurements.
- IDs as model inputs because they do not represent content quality.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]

print("Selected features:")
for f in feature_fields:
    print("-", f)

Selected features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- scroll_events


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [24]:
grain_check = con.execute("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM march_df
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

grain_check

,report_date,client_hash_id,content_hash_id,row_count


In [25]:
date_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM march_df
""").df()

date_check

,total_rows,first_date,last_date
0,10000,2026-03-01,2026-03-31


In [26]:
availability_check = con.execute("""
SELECT
    COUNT(*) AS gsc_available_rows
FROM march_df
WHERE gsc_data_available IS TRUE
""").df()

availability_check

,gsc_available_rows
0,3720


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
## Data Limits

This dataset supports observed performance analysis and decision support, but it cannot prove that a content update will directly improve rankings or traffic.

The data has limitations:
- The sample only represents March 2026 performance and may not capture all seasonal changes.
- Some clients do not have complete Search Console or Analytics history.
- Missing values may represent unavailable tracking data rather than zero performance.
- The dataset does not contain the actual result of a content refresh, so the analysis cannot measure causal impact.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
missing_check = march_df[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_sessions",
        "scroll_events"
    ]
].isna().mean()

missing_check


,0
gsc_impressions,0.0000
gsc_clicks,0.0000
gsc_avg_position,0.6280
ga4_sessions,0.3204
scroll_events,0.3204


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.